# <span style="color:#F57C00">Explaining the Model: SHAP and Ablation</span>

**What this notebook does.** The final model of the modeling notebook is one XGBoost model with 30 features. Two questions remain, and they need two different tools:

| Part | Question | How |
| :--- | :--- | :--- |
| **1. SHAP** | What does the model **use**, and how does it use the green share? | The saved model is explained as it is (SHAP values of every prediction). Nothing is trained. |
| **2. Ablation** | What does the model **need**? What is lost if a group of features is missing? | The model is trained again without one group at a time and tested with the two tests of the modeling notebook. |

The two parts answer different questions: SHAP tells what the finished model looks at, ablation tells what a model **loses** when the information is not there (other features can take over, so a feature that SHAP likes can still be replaceable).

**Read this first.** SHAP values and ablation both describe an *association* in the data, like the model itself. They do not show what happens when a district really gets more green; the what-if notebook changes the inputs and asks the model, and that is a different calculation.

**Output.** The ablation results are saved in `data/models/ablation.csv`; a second run reads them and takes about 1 minute (the first run takes about 15 minutes on 16 CPU cores).

# <span style="color:#F57C00">Part 1: SHAP, What the Final Model Uses</span>

**SHAP** (SHapley Additive exPlanations) splits every prediction into a starting value (the mean prediction) plus one contribution per feature, and the contributions add up exactly to the prediction. A feature with large contributions (up or down) is a feature that the model *uses*. For tree models SHAP is calculated exactly and fast (`shap.TreeExplainer`).

## Step 1: Setup
The model table is built by the same code as in the what-if notebook (a copy on purpose, so that this notebook can be read from top to bottom). A check against the model card stops the notebook if the features or the rows differ from the ones the model was trained with.

In [ ]:
import json
import warnings
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from joblib import Parallel, delayed
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.metrics import r2_score, root_mean_squared_error
from sklearn.model_selection import GroupKFold
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

PROCESSED = Path("../data/processed")
MODELS = Path("../data/models")
TARGET = "landsat_lst_c"  # the summer surface temperature of the built-up area (°C)

green = pd.read_csv(PROCESSED / "greenness_districts_summer_2013_2025.csv")
climate = pd.read_csv(PROCESSED / "climate_districts_monthly_2000_2025.csv")
population = pd.read_csv(PROCESSED / "tuik_districts_population_clean.csv")
static = pd.read_csv(PROCESSED / "districts_urban_static.csv")
districts = gpd.read_file(PROCESSED / "turkey_districts_master.geojson")
centers = pd.DataFrame(
    districts[["district_code", "center_lat", "center_lon"]]
)  # only the center points, without the polygons
print(
    f"greenness {green.shape} | climate {climate.shape} | population {population.shape} | static {static.shape} | centers {centers.shape}"
)

In [ ]:
summer = (
    climate[climate["month"].isin([7, 8])]
    .groupby(["district_code", "year"])
    .agg(temp_mean_ja=("temp_mean", "mean"), hot_days_ja=("hot_days", "sum"))
    .reset_index()
)
spring = (
    climate[climate["month"].between(3, 6)]
    .groupby(["district_code", "year"])
    .agg(precip_spring=("precip_sum", "sum"))
    .reset_index()
)
weather = summer.merge(
    spring, on=["district_code", "year"], validate="one_to_one"
)  # all districts and years

age_65_plus = [
    c
    for c in population.columns
    if c.startswith("total_")
    and c.split("_")[1][:2] in ("65", "70", "75", "80", "85", "90")
]
population["share65"] = population[age_65_plus].sum(axis=1) / population["total_pop"]
population["logpop"] = np.log(population["total_pop"])

table = (
    green.merge(
        weather, on=["district_code", "year"], how="left", validate="one_to_one"
    )
    .merge(
        population[["district_code", "year", "share65", "logpop"]],
        on=["district_code", "year"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        static[["district_code", "elevation_m", "built_share_2020"]],
        on="district_code",
        how="left",
        validate="many_to_one",
    )
    .merge(centers, on="district_code", how="left", validate="many_to_one")
)
assert len(table) == len(green), "a join must not change the number of rows"

REGIONS = {
    "Marmara": [10, 11, 16, 17, 22, 34, 39, 41, 54, 59, 77],
    "Aegean": [3, 9, 20, 35, 43, 45, 48, 64],
    "Mediterranean": [1, 7, 15, 31, 32, 33, 46, 80],
    "Central Anatolia": [6, 18, 26, 38, 40, 42, 50, 51, 58, 66, 68, 70, 71],
    "Black Sea": [5, 8, 14, 19, 28, 29, 37, 52, 53, 55, 57, 60, 61, 67, 69, 74, 78, 81],
    "Eastern Anatolia": [4, 12, 13, 23, 24, 25, 30, 36, 44, 49, 62, 65, 75, 76],
    "Southeastern Anatolia": [2, 21, 27, 47, 56, 63, 72, 73, 79],
}
region_of_city = {city: region for region, cities in REGIONS.items() for city in cities}

WEATHER = ["temp_mean_ja", "hot_days_ja", "precip_spring"]
WITHIN_CITY = [
    "landsat_green_share",
    "elevation_m",
    "built_share_2020",
    "log_urban_km2",
    "logpop",
    "share65",
    "temp_mean_ja",
    "hot_days_ja",
]


def add_within_city(frame, columns):
    # every value minus the mean of the same city in the same year
    frame = frame.copy()
    city_means = frame.groupby(["city_code", "year"])[columns].transform("mean")
    frame[[f"{name}_vs_city" for name in columns]] = (
        frame[columns] - city_means
    ).to_numpy()
    return frame


table["log_urban_km2"] = np.log(table["urban_km2"])
reliable = (
    (table["urban_km2"] >= 0.5)
    & (table["landsat_cover"] >= 0.5)
    & table[TARGET].notna()
)
table = table[reliable].dropna(subset=["share65", "logpop"])
table = table[
    table.groupby(["city_code", "year"])["district_code"].transform("size") >= 3
]

region_columns = pd.get_dummies(
    table["city_code"].map(region_of_city), prefix="region", dtype=int
)[[f"region_{name}" for name in REGIONS]]
table = add_within_city(
    pd.concat([table, region_columns], axis=1), WITHIN_CITY
).reset_index(drop=True)

FEATURES = (
    [
        "landsat_green_share",
        "elevation_m",
        "built_share_2020",
        "log_urban_km2",
        "logpop",
        "share65",
        "temp_mean_ja",
        "hot_days_ja",
        "precip_spring",
        "center_lat",
        "center_lon",
    ]
    + list(region_columns.columns)
    + [f"{name}_vs_city" for name in WITHIN_CITY]
)

# the two feature groups of Part 5 of the modeling notebook: clear sky and the climate normals of 2000-2012 (before the first summer of the table)
WITHIN_CITY_NEW = WITHIN_CITY + [
    "landsat_cover"
]  # the within-city columns of the final model also have the clear sky
early = climate[climate["year"] <= 2012]
normals = pd.DataFrame(
    {
        "precip_clim": early.groupby(["district_code", "year"])["precip_sum"]
        .sum()
        .groupby("district_code")
        .mean(),
        "temp_clim": early.groupby("district_code")["temp_mean"].mean(),
    }
).reset_index()
table = add_within_city(
    table.merge(normals, on="district_code", how="left", validate="many_to_one"),
    WITHIN_CITY_NEW,
)
ADDED_FEATURES = ["landsat_cover", "landsat_cover_vs_city", "precip_clim", "temp_clim"]
FEATURES = FEATURES + ADDED_FEATURES
MAX_GREEN = table[
    "landsat_green_share"
].max()  # the highest green share in the data: a tree model cannot predict beyond what it has seen
print(
    f"{len(table)} rows | {table['district_code'].nunique()} districts | {table['city_code'].nunique()} cities | {len(FEATURES)} features"
)

## Step 2: The final model and its model card
The small class of the modeling notebook is copied here: the final model is one XGBoost model that learns the surface temperature minus half of the air temperature of the summer and adds that half back when it predicts. The model card says which file and which share.

In [ ]:
GREEN = (
    "landsat_green_share",
    "landsat_green_share_vs_city",
)  # more green must never mean a hotter surface


def make_xgb(features, params, seed=0, n_jobs=1):
    # XGBoost with the settings of Part 4; the green share may only lower the prediction
    monotone = tuple(-1 if name in GREEN else 0 for name in features)
    return XGBRegressor(
        **params,
        monotone_constraints=monotone,
        tree_method="hist",
        n_jobs=n_jobs,
        random_state=seed
    )


class AirTemperatureOffset(RegressorMixin, BaseEstimator):
    """The final model. `estimator` learns the surface temperature minus a share of the air temperature of the summer;
    the share is added back when predicting. A hotter summer then moves the prediction directly, which trees cannot do
    on their own beyond the summers they have seen. `RegressorMixin` must come first: scikit-learn only recognizes a
    class as a regressor in that order, and cross-validation helpers check it. The state is explicit both ways:
    `fit` sets `estimator_`, and `from_loaded` sets it for a model that was read from a file."""

    def __init__(self, estimator, share=0.5, column="temp_mean_ja"):
        self.estimator, self.share, self.column = estimator, share, column

    def fit(self, X, y):
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        self.n_features_in_ = len(self.feature_names_in_)
        self.estimator_ = clone(self.estimator).fit(X, y - self.share * X[self.column])
        return self

    def predict(self, X):
        estimator = getattr(self, "estimator_", None)  # a model loaded from a file has no fitted state yet
        if estimator is None:
            raise ValueError("the model is neither fitted nor loaded")
        if list(X.columns) != list(self.feature_names_in_):
            raise ValueError("the prediction columns differ from the fitted model")
        return estimator.predict(X) + self.share * X[self.column].to_numpy()

    @classmethod
    def from_loaded(cls, estimator, features, share=0.5, column="temp_mean_ja"):
        """A native XGBoost file read from disk, with its fitted state made explicit."""
        booster_names = estimator.get_booster().feature_names
        if booster_names is not None and list(booster_names) != list(features):
            raise ValueError("the model file uses other feature names than the model card")
        model = cls(estimator, share, column)
        model.estimator_ = estimator
        model.feature_names_in_ = np.asarray(features, dtype=object)
        model.n_features_in_ = len(features)
        return model


def new_final(features, params, seed=0, n_jobs=1):
    return AirTemperatureOffset(make_xgb(features, params, seed, n_jobs))


def load_final(card, folder):
    # the final model from the files of the model card, without training
    trees = XGBRegressor()
    trees.load_model(folder / card["model_file"])  # the native format of XGBoost
    return AirTemperatureOffset.from_loaded(
        trees, card["features"], card["air_temperature_share"],
        card["air_temperature_column"],
    )

In [ ]:
card = json.loads((MODELS / "model_card.json").read_text())
assert (
    "air_temperature_share" in card
), "this notebook is written for the final model of the modeling notebook"
assert (
    FEATURES == card["features"]
), "the features of this notebook differ from the model card"
assert (
    len(table) == card["training_rows"]
), "the model table differs from the one the model was trained on"
final = load_final(card, MODELS)
share, air = card["air_temperature_share"], card["air_temperature_column"]
print(
    f"{card['algorithm']}, {len(FEATURES)} features, {len(table)} rows ({card['trained_on_years'][0]}-{card['trained_on_years'][1]})"
)

## Step 3: The SHAP values
`shap.TreeExplainer` explains the trees of the model, and the trees learned the surface temperature **minus half of the air temperature**. The other half is a straight line in the air temperature, so it is added to the contribution of `temp_mean_ja` (`share × (air temperature − its mean)`) and to the starting value; then the contributions add up to the real prediction of the final model, which a check confirms. All 10,622 rows of the model table are explained.

In [ ]:
explainer = shap.TreeExplainer(final.estimator)  # the trees of the final model
values = pd.DataFrame(
    explainer.shap_values(table[FEATURES]), columns=FEATURES, index=table.index
)
values[air] += share * (
    table[air] - table[air].mean()
)  # the straight part of the model belongs to the air temperature
base_value = (
    float(np.ravel(explainer.expected_value)[0]) + share * table[air].mean()
)  # the starting value: the mean prediction

prediction = final.predict(table[FEATURES])
assert (
    np.abs(base_value + values.sum(axis=1) - prediction).max() < 1e-3
), "the contributions must add up to the prediction"
print(
    f"starting value {base_value:.2f} °C | the contributions add up to the prediction (largest difference {np.abs(base_value + values.sum(axis=1) - prediction).max():.1e} °C)"
)

## Step 4: Which features does the model use?
The mean of the absolute SHAP value of a feature is its **importance**: how many °C it moves a prediction on average. The first figure shows it for the 12 most used features (left) and, added up, for the 7 groups of features that Part 2 removes one by one (right). The second figure shows every district-summer as a point: the position is the contribution, the colour is the value of the feature (red = high).

In [ ]:
GROUPS = {
    "green share": ["landsat_green_share", "landsat_green_share_vs_city"],
    "weather of the summer": [
        "temp_mean_ja",
        "hot_days_ja",
        "precip_spring",
        "temp_mean_ja_vs_city",
        "hot_days_ja_vs_city",
    ],
    "climate normals": ["precip_clim", "temp_clim"],
    "location": ["center_lat", "center_lon"]
    + [name for name in FEATURES if name.startswith("region_")],
    "buildings and terrain": [
        "elevation_m",
        "built_share_2020",
        "log_urban_km2",
        "elevation_m_vs_city",
        "built_share_2020_vs_city",
        "log_urban_km2_vs_city",
    ],
    "population": ["logpop", "share65", "logpop_vs_city", "share65_vs_city"],
    "clear sky": ["landsat_cover", "landsat_cover_vs_city"],
}
assert sorted(sum(GROUPS.values(), [])) == sorted(
    FEATURES
), "every feature must be in exactly one group"

importance = values.abs().mean().sort_values(ascending=False)
by_group = pd.Series(
    {name: importance[columns].sum() for name, columns in GROUPS.items()}
).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), gridspec_kw={"width_ratios": [1, 1]})
importance.head(12).iloc[::-1].plot.barh(ax=axes[0], color="#F57C00")
axes[0].set_xlabel("mean |SHAP value| (°C)")
axes[0].set_title("The 12 most used features")
(by_group / by_group.sum() * 100).iloc[::-1].plot.barh(ax=axes[1], color="#2c6e91")
axes[1].set_xlabel("share of the total importance (%)")
axes[1].set_title("The same, by group of features")
plt.tight_layout()
plt.show()

shap.summary_plot(
    values.to_numpy(), table[FEATURES], max_display=12, show=False
)  # every district-summer is a point
plt.tight_layout()
plt.show()
print((by_group / by_group.sum() * 100).round(1).to_dict())

## Step 5: How does the model use the green share?
The model has two green columns: the green share itself and the share minus the mean of the city (`landsat_green_share_vs_city`). The sum of their contributions is what the model "says" about the green share of a district. The figure shows it against the green share. The what-if of the other notebook raises the green share of **one** district and lets the model predict again; the figure below is different: it compares *different districts* with each other.

In [ ]:
green_effect = values[GROUPS["green share"]].sum(
    axis=1
)  # the contribution of both green columns
slope = (
    np.polyfit(table["landsat_green_share"], green_effect, 1)[0] * 0.10
)  # °C per +10 points of green share, across the districts
corr = np.corrcoef(table["landsat_green_share"], green_effect)[0, 1]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].scatter(
    table["landsat_green_share"] * 100, green_effect, s=4, alpha=0.3, color="#7b7b7b"
)
line = np.array([0, table["landsat_green_share"].max()])
axes[0].plot(
    line * 100,
    np.polyval(np.polyfit(table["landsat_green_share"], green_effect, 1), line),
    color="#F57C00",
    lw=2,
    label=f"straight line: {slope:+.2f} °C per +10 points",
)
axes[0].axhline(0, color="grey", lw=0.8)
axes[0].set_xlabel("green share (%)")
axes[0].set_ylabel("contribution of the two green columns (°C)")
axes[0].set_title("Every point is a district-summer")
axes[0].legend()

quarters = pd.qcut(
    table["landsat_green_share"],
    4,
    labels=["least green 25%", "second", "third", "greenest 25%"],
)
spread = green_effect.groupby(quarters).agg(["mean", "std"])
spread["mean"].plot.bar(
    ax=axes[1], yerr=spread["std"], color="#F57C00", rot=0, capsize=4
)
axes[1].set_xlabel("green share, in four groups of equal size")
axes[1].set_ylabel("mean contribution (°C), bar = standard deviation")
axes[1].set_title("Less green: the model adds heat; more green: it takes heat away")
plt.tight_layout()
plt.show()
fit = np.poly1d(np.polyfit(table["landsat_green_share"], green_effect, 1))
print(
    f"correlation of the green contribution with the green share: {corr:+.2f} | across the districts {slope:+.2f} °C per +10 points (what-if notebook, one district at a time: {card['what_if_plus_10_points_green_mean_change_c']:+.2f})"
)
print(
    f"the straight line goes from {fit(0):+.1f} °C at 0% green to {fit(table['landsat_green_share'].max()):+.1f} °C at {table['landsat_green_share'].max():.0%}; mean contribution of the four groups: { {k: round(float(v), 2) for k, v in spread['mean'].items()} }"
)

**What we learned (Part 1)**
- **The air temperature of the summer is used most:** its mean |SHAP value| is 1.44 °C (the next feature has 0.68 °C); together the weather columns are **37%** of the total importance. A hot summer pushes the prediction up by up to about 5 °C (red points on the right of the second figure).
- **The green share is second:** 0.68 °C, **17.5%** for the two green columns together. More green pushes the prediction down (the red points of `landsat_green_share` lie on the left). Then come the clear sky (0.38 °C: a low share of clear pixels means a cloud remnant, and the surface reads cooler, as in the EDA), the elevation, the longitude, the temperature normal and the hot days (about 0.2 °C each).
- **The population columns are hardly used:** four features share 3.6% of the importance. Part 2 asks whether they are needed at all.
- **The green share acts almost like a straight line across the districts:** the contribution of the two green columns has a correlation of -0.97 with the green share and goes from about +1.6 °C at 0% green to -2.8 °C at 92%; the four groups of equal size have mean contributions of +1.12, +0.41, -0.32 and -1.28 °C. The slope is **-0.49 °C per +10 points** across the districts.
- **This is not the what-if effect:** the what-if notebook raises the green share of **one** district and finds -0.63 °C per +10 points. SHAP shares the credit of a prediction between features that move together (districts with more green also differ in other ways), so its slope is a description of how the model explains the differences between districts, not the answer to "what if this district got greener". Use the what-if for that.

# <span style="color:#F57C00">Part 2: Ablation, What the Model Needs</span>

**Ablation** removes information and measures the loss. The 30 features are in the 7 groups of Part 1. For every group the final model is trained **again without the group** (same settings, same monotone rule), and tested with the two tests of the modeling notebook on the same 4 fresh seeds:
- **new cities:** 5-fold cross-validation that keeps whole cities together;
- **next summers:** train on the years before *t*, predict *t*, for *t* = 2019, 2020, 2021, 2022.

Both are scored with `balance = r2_level + r2_within` (the city level plus the differences between the districts of a city, which is what the what-if is about); the **joint score** is the mean of the two tests. A bootstrap over the 80 cities (1,000 draws, the `sample_weight` of the scikit-learn metrics) gives a 95% interval of the **loss of the joint score**: a loss counts when the interval stays above 0.

Three notes. **1.** If the group with the air temperature (`weather of the summer`) is removed, the air temperature offset has nothing to subtract, so that model is a plain XGBoost model. **2.** The row `without the air temperature offset` keeps all 30 features and drops only the offset (the step of Part 5 of the modeling notebook). **3.** The other features can partly take over the information of a removed group (location, climate normals and elevation are related), so the loss of a group is **smaller than its SHAP importance** suggests when others can replace it.

## Step 1: The two tests as functions
The same functions as in the modeling notebook, written for any list of columns; `make(seed)` returns a new, untrained model.

In [ ]:
TRAIN_END = 2022  # the last training year; 2023-2025 are not used here
FRESH_SEEDS = (200, 201, 202, 203)
FOLDS, NEXT_YEARS = 5, (2019, 2020, 2021, 2022)
train = table[table["year"] <= TRAIN_END].reset_index(drop=True)
next_rows = pd.concat([train[train["year"] == t] for t in NEXT_YEARS]).reset_index(
    drop=True
)  # in the order of the predictions


def scores(rows, predicted):
    # rows: the districts that were predicted; predicted: the predictions in the same order
    result = rows[["city_code", "year", TARGET]].assign(predicted=np.asarray(predicted))
    city_year = result.groupby(["city_code", "year"])[[TARGET, "predicted"]]
    level = city_year.mean()  # one value per city and year
    inside = result[[TARGET, "predicted"]] - city_year.transform(
        "mean"
    )  # the part inside the cities
    r2_level, r2_within = r2_score(level[TARGET], level["predicted"]), r2_score(
        inside[TARGET], inside["predicted"]
    )
    return pd.Series(
        {
            "rmse": root_mean_squared_error(result[TARGET], result["predicted"]),
            "r2_level": r2_level,
            "r2_within": r2_within,
            "balance": r2_level + r2_within,
        }
    )


def predict_new_cities(make, columns, seed):
    # test 1: out-of-fold predictions; every city is predicted by a model that has not seen it
    pred, fold = np.empty(len(train)), np.empty(len(train), dtype=int)
    for k, (fit_index, test_index) in enumerate(
        GroupKFold(n_splits=FOLDS, shuffle=True, random_state=seed).split(
            train, groups=train["city_code"]
        )
    ):
        pred[test_index] = (
            make(seed)
            .fit(train.iloc[fit_index][columns], train.iloc[fit_index][TARGET])
            .predict(train.iloc[test_index][columns])
        )
        fold[test_index] = k
    return pred, fold


def predict_next_summers(make, columns, seed):
    # test 2: every summer is predicted by a model that was trained on the years before it
    out = []
    for year in NEXT_YEARS:
        earlier, current = train[train["year"] < year], train[train["year"] == year]
        out.append(
            make(seed).fit(earlier[columns], earlier[TARGET]).predict(current[columns])
        )
    return np.concatenate(out)


def joint_score(pred_new, fold, pred_next):
    new_cities = np.mean(
        [scores(train[fold == k], pred_new[fold == k])["balance"] for k in range(FOLDS)]
    )
    return (new_cities + scores(next_rows, pred_next)["balance"]) / 2

## Step 2: The variants
Every variant is a list of columns and a way to build the model: the final model (`AirTemperatureOffset` around the trees), or plain trees when the air temperature is not there.

In [ ]:
PARAMS = card["settings"]  # the settings of the production model (the model card)


def with_offset(columns):
    return lambda seed: new_final(columns, PARAMS, seed)


def plain(columns):
    return lambda seed: make_xgb(columns, PARAMS, seed)


variants = {"the final model (all 30 features)": (FEATURES, with_offset(FEATURES))}
for name, columns in GROUPS.items():
    kept = [f for f in FEATURES if f not in columns]
    variants[f"without: {name}"] = (
        kept,
        plain(kept) if air in columns else with_offset(kept),
    )  # no air temperature, no offset
variants["without the air temperature offset"] = (FEATURES, plain(FEATURES))
pd.Series(
    {name: len(columns) for name, (columns, make) in variants.items()}, name="features"
)

## Step 3: The ablation
The tests of every variant run in parallel (one thread per model). Besides the scores, two more numbers are saved per variant: the **+10 points effect** (the model is trained on all years, the green share of every district of the latest summer rises by 10 points, like the what-if check of the modeling notebook; empty where the green columns are removed) and the **loss against the final model** with its bootstrap interval. The results are saved in `data/models/ablation.csv`; when the file exists it is only read.

In [ ]:
cities = np.sort(train["city_code"].unique())
counts = np.random.default_rng(0).multinomial(
    len(cities), np.ones(len(cities)) / len(cities), size=1000
)
counts = np.vstack(
    [np.ones(len(cities)), counts]
)  # row 0: every city once = the plain score


def sample_scores(rows, predicted):
    # balance, within-city R2 and RMSE of the predictions for every sample of cities (rows of `counts`); a city that is drawn twice counts twice
    r = rows[["city_code", "year", TARGET]].assign(predicted=np.asarray(predicted))
    grouped = r.groupby(["city_code", "year"])[[TARGET, "predicted"]]
    level = grouped.mean().reset_index()
    inside = r[[TARGET, "predicted"]] - grouped.transform("mean")
    city_of_level, city_of_row = np.searchsorted(
        cities, level["city_code"]
    ), np.searchsorted(cities, r["city_code"])
    out = []
    for weights in counts:
        w_level, w_row = weights[city_of_level], weights[city_of_row]
        r2_within = r2_score(inside[TARGET], inside["predicted"], sample_weight=w_row)
        out.append(
            [
                r2_score(level[TARGET], level["predicted"], sample_weight=w_level)
                + r2_within,
                r2_within,
                root_mean_squared_error(r[TARGET], r["predicted"], sample_weight=w_row),
            ]
        )
    return np.array(out)


def plus_10_effect(columns, make):
    # the model is trained on all years; the green share of every district of the latest summer rises by 10 points (one district at a time)
    if "landsat_green_share" not in columns:
        return np.nan
    model = make(0).fit(table[columns], table[TARGET])
    latest = table[table["year"] == table["year"].max()]
    n_districts = latest.groupby("city_code")["district_code"].transform("size")
    changed = latest[columns].copy()
    changed["landsat_green_share"] += 0.10
    changed["landsat_green_share_vs_city"] += 0.10 * (
        1 - 1 / n_districts
    )  # the city mean moves by 1/n of the change
    return float((model.predict(changed) - model.predict(latest[columns])).mean())


def evaluate_all():
    # every (variant, seed, test) is one task, all of them run at the same time (one thread per model)
    tasks = [
        (name, i, test)
        for name in variants
        for i in range(len(FRESH_SEEDS))
        for test in ("new", "next")
    ]

    def run(name, i, test):
        columns, make = variants[name]
        return (
            predict_new_cities(make, columns, FRESH_SEEDS[i])
            if test == "new"
            else predict_next_summers(make, columns, FRESH_SEEDS[i])
        )

    results = Parallel(n_jobs=-1, backend="threading")(
        delayed(run)(*task) for task in tasks
    )
    runs = {
        name: dict(
            pred_new=[None] * len(FRESH_SEEDS),
            fold=[None] * len(FRESH_SEEDS),
            pred_next=[None] * len(FRESH_SEEDS),
        )
        for name in variants
    }
    for (name, i, test), result in zip(tasks, results):
        if test == "new":
            runs[name]["pred_new"][i], runs[name]["fold"][i] = result
        else:
            runs[name]["pred_next"][i] = result
    return runs


def ablation():
    path = MODELS / "ablation.csv"
    if path.exists():
        return pd.read_csv(path, index_col=0)
    runs = evaluate_all()
    full = "the final model (all 30 features)"
    sampled = {  # variant -> test -> scores of every sample of cities, the mean over the seeds
        name: {
            "new cities": np.mean(
                [
                    sample_scores(train, run["pred_new"][i])
                    for i in range(len(FRESH_SEEDS))
                ],
                axis=0,
            ),
            "next summers": np.mean(
                [
                    sample_scores(next_rows, run["pred_next"][i])
                    for i in range(len(FRESH_SEEDS))
                ],
                axis=0,
            ),
        }
        for name, run in runs.items()
    }
    rows = {}
    for name, run in runs.items():
        loss = {
            t: sampled[full][t] - sampled[name][t]
            for t in ("new cities", "next summers")
        }  # draws x (balance, within-city R2, RMSE); the RMSE rises when information is lost
        joint_loss = (loss["new cities"][:, 0] + loss["next summers"][:, 0]) / 2
        seeds = [
            joint_score(run["pred_new"][i], run["fold"][i], run["pred_next"][i])
            for i in range(len(FRESH_SEEDS))
        ]
        rows[name] = {
            "features": len(variants[name][0]),
            "joint score": np.mean(seeds),
            "new cities": np.mean(
                [
                    np.mean(
                        [
                            scores(
                                train[run["fold"][i] == k],
                                run["pred_new"][i][run["fold"][i] == k],
                            )["balance"]
                            for k in range(FOLDS)
                        ]
                    )
                    for i in range(len(FRESH_SEEDS))
                ]
            ),
            "next summers": np.mean(
                [
                    scores(next_rows, run["pred_next"][i])["balance"]
                    for i in range(len(FRESH_SEEDS))
                ]
            ),
            "RMSE new cities": np.mean(
                [
                    scores(train, run["pred_new"][i])["rmse"]
                    for i in range(len(FRESH_SEEDS))
                ]
            ),
            "RMSE next summers": np.mean(
                [
                    scores(next_rows, run["pred_next"][i])["rmse"]
                    for i in range(len(FRESH_SEEDS))
                ]
            ),
            "loss of the joint score": joint_loss[0],
            "loss 95% from": np.percentile(joint_loss[1:], 2.5),
            "loss 95% to": np.percentile(joint_loss[1:], 97.5),
            "loss of the within-city R2 (new cities)": (
                sampled[full]["new cities"][0, 1] - sampled[name]["new cities"][0, 1]
            ),
            "loss of the within-city R2 (next summers)": (
                sampled[full]["next summers"][0, 1]
                - sampled[name]["next summers"][0, 1]
            ),
            "+10 points effect": plus_10_effect(*variants[name]),
        }
    table_ = pd.DataFrame(rows).T
    table_.round(6).to_csv(path)
    return table_


result = ablation()
result.round(3)

## Step 4: What is lost?
The figure shows the loss of the joint score of every variant with its 95% interval (a bar to the right = the model gets worse without the information). The second figure shows the +10 points effect of the variants that still have the green columns.

In [ ]:
shown = result.drop(index="the final model (all 30 features)").sort_values(
    "loss of the joint score"
)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
axes[0].barh(
    shown.index,
    shown["loss of the joint score"],
    xerr=[
        shown["loss of the joint score"] - shown["loss 95% from"],
        shown["loss 95% to"] - shown["loss of the joint score"],
    ],
    color="#F57C00",
    capsize=3,
)
axes[0].axvline(0, color="black", lw=0.8)
axes[0].set_xlabel(
    "loss of the joint score (the final model: %.3f)"
    % result.loc["the final model (all 30 features)", "joint score"]
)
axes[0].set_title("What the model loses without the information")
effect = result["+10 points effect"].dropna()
axes[1].barh(effect.index, effect, color="#2c6e91")
axes[1].axvline(
    card["what_if_plus_10_points_green_mean_change_c"],
    color="black",
    ls="--",
    lw=1,
    label="model card",
)
axes[1].set_xlabel("+10 points of green share: mean change (°C), latest summer")
axes[1].set_title("The what-if effect of the variants")
axes[1].legend(loc="lower left")
plt.tight_layout()
plt.show()
print(
    shown[
        ["loss of the joint score", "loss 95% from", "loss 95% to", "+10 points effect"]
    ]
    .round(3)
    .to_string()
)

**What we learned (Part 2)**
- **The green share is the information that costs the most to lose:** without the two green columns the joint score falls by **0.083** (95% interval 0.051 to 0.116); the within-city R² of the new-cities test falls by 0.107, and the RMSE rises from 1.71 to 1.84 °C (new cities) and from 1.36 to 1.43 °C (next summers). No other group makes up for it.
- **The weather of the summer comes second (0.050, 0.027 to 0.073), although SHAP puts it first** (37% against 17.5%): without the air temperature other columns take over a large part of its information (the climate normals, the elevation, the location, the clear sky), so a feature that the model *uses* most is not the one that is hardest to *replace*. The RMSE still rises to 1.85 °C (new cities) and 1.47 °C (next summers).
- **The clear sky (0.023, 0.013 to 0.035), the climate normals (0.017, 0.012 to 0.022) and the location (0.017, 0.006 to 0.027) each cost a clear, small amount.** The clear sky and the climate normals are the two groups added in Part 5 of the modeling notebook, so their loss confirms that those additions help.
- **The buildings and terrain (0.010, interval -0.002 to 0.020) are not clearly needed, and the population columns are not needed at all** (-0.002, interval -0.006 to 0.003: the model is as good without them). This agrees with the EDA, where the population hardly explained the surface temperature. They stay in the model because they do no harm; a simpler model without the four population columns would be as accurate.
- **The air temperature offset is a small step on its own (0.007, interval 0.000 to 0.014),** as in Part 5 of the modeling notebook; its value is in the hot summers of the exam, not in these two tests.
- **The what-if effect is stable, with one exception:** the +10 points effect of the final model is -0.64 °C (latest summer). Without the buildings, the climate normals or the population it stays between -0.64 and -0.68 °C; without the location -0.70, without the clear sky -0.73. **Without the weather of the summer it grows to -0.88 °C**: then the green share has to carry the differences of climate between the cities, and the model reads them as a cooling effect of green. This supports the reading of the modeling notebook that the climate columns take over a part of the difference between cities that the green share would otherwise pick up.
- **Limits:** a group that others can replace shows a small loss even if it is informative; and all numbers are associations of the data, like the model.